# LeetCode 423 — Reconstruct Original Digits from English bằng Python

Link đề: https://leetcode.com/problems/reconstruct-original-digits-from-english/ · Độ khó: **Medium** · Chủ đề: Hash Table, Math, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def originalDigits(self, s: str) -> str:
        ...
```

Ghi chú: phần phát biểu đề dịch sang tiếng Việt từ đề gốc; các ví dụ đều được chạy lại bằng code trong notebook.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi `s` là các từ tiếng Anh của chữ số `0..9` (`zero`, `one`, ..., `nine`) bị **xáo trộn và dính liền**; hãy khôi phục các chữ số và in chúng theo **thứ tự tăng dần**.
2. Nắm công cụ Python: `collections.Counter`, truy cập `counter['z']` (mặc định `0` nếu thiếu), nhân chuỗi `'3' * 2 == '33'`, `''.join(...)`, danh sách tra cứu và `zip`/`enumerate`.
3. Đi từ **thử mọi cách chia (quay lui)** → **tìm chữ cái "độc quyền" của từng chữ số và bóc dần (công thức)** → **cùng ý bóc nhưng chạy theo bảng và trừ trọn từ** → **hệ phương trình tuyến tính trên số đếm chữ cái (khử Gauss, `Fraction`)**.
4. Hiểu mấu chốt: `z` chỉ có trong `zero`, `w` chỉ trong `two`, `u` chỉ trong `four`, `x` chỉ trong `six`, `g` chỉ trong `eight`; bóc chúng ra thì các chữ cái khác trở nên "độc quyền" ở bước kế tiếp (tam giác hóa hệ).
5. Tránh các lỗi kinh điển: tính theo thứ tự sai (dùng giá trị chưa biết), quên trừ phần chồng lấn (`h` có ở cả `three` và `eight`), ghép kết quả không theo thứ tự `0..9`, và greedy cắt từ theo thứ tự `0..9` (có thể "cướp" chữ cái của từ khác).

## 2. Đề bài & ví dụ

Cho chuỗi `s` chứa **biểu diễn tiếng Anh bị xáo trộn thứ tự** của các chữ số `0-9`. Trả về các chữ số theo **thứ tự tăng dần** (dưới dạng một chuỗi).

**Ví dụ 1:** `s = "owoztneoer"` → `"012"`. (Các từ `zero`, `one`, `two` bị trộn: `owoztneoer`.)

**Ví dụ 2:** `s = "fviefuro"` → `"45"`. (`four` và `five`.)

**Ràng buộc:** `1 <= s.length <= 10^5`; `s[i]` thuộc tập `["e","g","f","i","h","o","n","s","r","u","t","w","v","x","z"]`; `s` được **đảm bảo hợp lệ** (luôn ghép được từ các từ chữ số).

Bảng chữ số:

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|
| zero | one | two | three | four | five | six | seven | eight | nine |

In [1]:
from collections import Counter
from fractions import Fraction
from typing import List
import random
import time

WORDS = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]

examples = [
    ("owoztneoer", "012"),
    ("fviefuro", "45"),
]
for s, expected in examples:
    print(repr(s), "-> kỳ vọng", repr(expected))
print(WORDS)

'owoztneoer' -> kỳ vọng '012'
'fviefuro' -> kỳ vọng '45'
['zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine']


### 2.1. Đọc kỹ đề: các điểm cần để ý

- **Kết quả theo thứ tự tăng dần** — không phải thứ tự xuất hiện trong `s`, cũng không phải thứ tự ta "tìm ra" từng chữ số.
- **Một chữ số có thể xuất hiện nhiều lần** (ví dụ `s` gồm hai từ `two` ⇒ kết quả `"22"`).
- **Đảm bảo hợp lệ** nghĩa là ta không cần kiểm tra lỗi: số đếm ra luôn không âm và các chữ cái dùng hết đúng bằng 0.
- Chữ cái chỉ gồm 15 loại: `e g f i h o n s r u t w v x z` (đủ cho 10 từ).
- Có **nhiều cách chia** nhìn có vẻ hợp lý (ví dụ `one` có thể "mượn" `o` của `two`), nhưng nghiệm hợp lệ là **duy nhất**; hôm nay ta sẽ chứng minh điều đó bằng đại số.

In [2]:
print(Counter("owoztneoer"))
print("tập chữ cái của 10 từ:", sorted(set("".join(WORDS))))
print("số chữ cái khác nhau:", len(set("".join(WORDS))))

Counter({'o': 3, 'e': 2, 'w': 1, 'z': 1, 't': 1, 'n': 1, 'r': 1})
tập chữ cái của 10 từ: ['e', 'f', 'g', 'h', 'i', 'n', 'o', 'r', 's', 't', 'u', 'v', 'w', 'x', 'z']
số chữ cái khác nhau: 15


## 3. Kiến thức nền

### 3.1. `Counter` — đếm tần suất ký tự

`Counter("hello")` trả về một đối tượng giống `dict`: `{'l': 2, 'h': 1, 'e': 1, 'o': 1}`. Tra một khóa **không có** thì cho `0` chứ không báo lỗi (khác `dict` thường).

In [3]:
c = Counter("owoztneoer")
print(c)
print("c['o'] =", c["o"], "| c['z'] =", c["z"], "| c['x'] =", c["x"], "(không có -> 0)")
d = {"a": 1}
try:
    print(d["b"])
except KeyError as e:
    print("dict thường: KeyError", e)

Counter({'o': 3, 'e': 2, 'w': 1, 'z': 1, 't': 1, 'n': 1, 'r': 1})
c['o'] = 3 | c['z'] = 1 | c['x'] = 0 (không có -> 0)
dict thường: KeyError 'b'


### 3.2. Nhân chuỗi và `join`

`'3' * 2` cho `'33'`; `'3' * 0` cho `''` (chuỗi rỗng). Nhờ vậy `str(d) * cnt[d]` tạo đúng `cnt[d]` ký tự chữ số `d`, kể cả khi `cnt[d] = 0`. `''.join(list)` ghép các chuỗi trong danh sách.

In [4]:
print(repr("3" * 2), repr("3" * 0), repr("12" * 3))
cnt = [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
print("".join(str(d) * cnt[d] for d in range(10)))       # '012'

'33' '' '121212'
012


### 3.3. Danh sách tra cứu và `enumerate`

`WORDS[d]` là từ của chữ số `d`. `enumerate(WORDS)` cho các cặp `(d, word)`. Ta sẽ dùng để dựng bảng đếm chữ cái cho từng từ.

In [5]:
for d, w in enumerate(WORDS[:4]):
    print(d, w, dict(Counter(w)))

0 zero {'z': 1, 'e': 1, 'r': 1, 'o': 1}
1 one {'o': 1, 'n': 1, 'e': 1}
2 two {'t': 1, 'w': 1, 'o': 1}
3 three {'t': 1, 'h': 1, 'r': 1, 'e': 2}


### 3.4. Mỗi chữ cái "thuộc về" những chữ số nào?

Lập bảng nghịch đảo: với mỗi chữ cái, liệt kê các từ chứa nó. Chữ cái nào chỉ thuộc **một** từ thì gặp nó ta biết chắc từ đó xuất hiện (đây là chữ cái "độc quyền").

In [6]:
owners = {}
for d, w in enumerate(WORDS):
    for ch in set(w):
        owners.setdefault(ch, []).append(d)
for ch in sorted(owners):
    tag = "  <-- ĐỘC QUYỀN" if len(owners[ch]) == 1 else ""
    print(f"{ch}: {owners[ch]}{tag}")

e: [0, 1, 3, 5, 7, 8, 9]
f: [4, 5]
g: [8]  <-- ĐỘC QUYỀN
h: [3, 8]
i: [5, 6, 8, 9]
n: [1, 7, 9]
o: [0, 1, 2, 4]
r: [0, 3, 4]
s: [6, 7]
t: [2, 3, 8]
u: [4]  <-- ĐỘC QUYỀN
v: [5, 7]
w: [2]  <-- ĐỘC QUYỀN
x: [6]  <-- ĐỘC QUYỀN
z: [0]  <-- ĐỘC QUYỀN


Kết quả cho thấy chữ cái độc quyền là: `z` (chỉ `zero`), `w` (chỉ `two`), `u` (chỉ `four`), `x` (chỉ `six`), `g` (chỉ `eight`). Đây là mỏ neo đầu tiên; các chữ cái còn lại phải "bóc" dần.

## 4. Ý tưởng nền tảng: tính tay một ví dụ

Lấy `s = "owoztneoer"`. Đếm chữ cái: `o:3, w:1, z:1, t:1, n:1, e:2, r:1`.

- Có `z = 1` ⇒ chắc chắn có đúng **1** chữ số `0` (`zero`). Dùng hết: `z, e, r, o` ⇒ còn `o:2, w:1, t:1, n:1, e:1`.
- Có `w = 1` ⇒ có **1** chữ số `2` (`two`). Dùng `t, w, o` ⇒ còn `o:1, n:1, e:1`.
- Còn `o, n, e` ⇒ đúng một từ `one` ⇒ chữ số `1`.

Kết quả sắp theo thứ tự tăng dần: `"012"`. Nhận xét: ta **không cần thử mọi cách chia**, chỉ cần đi theo chữ cái độc quyền, rồi dùng kết quả đã biết để làm sạch các chữ cái khác.

In [7]:
c = Counter("owoztneoer")
def remove_word(counter, word, times):
    for ch in word:
        counter[ch] -= times

remove_word(c, "zero", c["z"])         # có 1 chữ z -> 1 từ zero
print("sau khi bóc zero:", {k: v for k, v in c.items() if v})
remove_word(c, "two", c["w"])
print("sau khi bóc two :", {k: v for k, v in c.items() if v})
print("còn lại:", {k: v for k, v in c.items() if v}, "= one")

sau khi bóc zero: {'o': 2, 'w': 1, 't': 1, 'n': 1, 'e': 1}
sau khi bóc two : {'o': 1, 'n': 1, 'e': 1}
còn lại: {'o': 1, 'n': 1, 'e': 1} = one


## 5. Cách 1 — Brute Force: thử mọi cách chia (quay lui)

**Ý tưởng:** gọi `cnt[d]` là số lần chữ số `d` xuất hiện. Thử lần lượt `d = 0, 1, ..., 9`: chọn `cnt[d]` từ `0` tới tối đa cho phép bởi số chữ cái hiện có, trừ các chữ cái đó, đệ quy sang `d + 1`. Khi hết cả 10 chữ số mà tất cả chữ cái đã về `0` ⇒ tìm được cách chia.

Cách này **luôn đúng** (vì thử hết) nhưng bùng nổ số trường hợp. Ta dùng nó như **oracle độc lập** trên chuỗi ngắn.

In [8]:
def digits_brute(s):
    letters = "efihognsrutwvxz"
    need = [Counter(w) for w in WORDS]
    have = Counter(s)

    def dfs(d, cnt):
        if d == 10:
            return cnt if all(v == 0 for v in have.values()) else None
        # số lần tối đa của chữ số d mà các chữ cái còn lại cho phép
        limit = min(have[ch] // k for ch, k in need[d].items())
        for c in range(limit + 1):
            for ch, k in need[d].items():
                have[ch] -= k * c
            res = dfs(d + 1, cnt + [c])
            for ch, k in need[d].items():
                have[ch] += k * c
            if res is not None:
                return res
        return None

    cnt = dfs(0, [])
    return "".join(str(d) * cnt[d] for d in range(10))


for s, expected in examples:
    got = digits_brute(s)
    print(repr(s), "->", repr(got), "| kỳ vọng", repr(expected))
    assert got == expected

'owoztneoer' -> '012' | kỳ vọng '012'
'fviefuro' -> '45' | kỳ vọng '45'


**Độ phức tạp:** số tổ hợp `(c_0, ..., c_9)` có thể rất lớn (tích của các giới hạn) — trong xấu nhất **mũ** theo độ dài `s`. Space `O(10)` cho ngăn xếp đệ quy.

**Nút thắt:** với `s` dài 100 000 ký tự, vét cạn hoàn toàn không khả thi. Hãy đo để thấy độ bùng nổ.

In [9]:
def make_valid(digits, rng):
    letters = list("".join(WORDS[d] for d in digits))
    rng.shuffle(letters)
    return "".join(letters)

rng = random.Random(1)
for L in [3, 6, 9, 12]:
    digits = [rng.randrange(10) for _ in range(L)]
    s = make_valid(digits, rng)
    t = time.perf_counter()
    r = digits_brute(s)
    print(f"{L:>2} chữ số, |s| = {len(s):>2}: {time.perf_counter() - t:.4f} giây")

 3 chữ số, |s| = 10: 0.0001 giây
 6 chữ số, |s| = 23: 0.0005 giây
 9 chữ số, |s| = 40: 0.0026 giây
12 chữ số, |s| = 50: 0.0030 giây


## 6. Ý tưởng thuật toán / bước đột phá

**Tam giác hóa bằng chữ cái độc quyền.** Gọi `c[x]` là số lần chữ cái `x` xuất hiện trong `s` và `cnt[d]` là số lần chữ số `d`. Mỗi chữ cái cho một **phương trình**: `c[x] = Σ (số lần x trong từ d) * cnt[d]`.

Các phương trình có chữ cái độc quyền cho nghiệm ngay:

- `c['z'] = cnt[0]`
- `c['w'] = cnt[2]`
- `c['u'] = cnt[4]`
- `c['x'] = cnt[6]`
- `c['g'] = cnt[8]`

Sau khi biết các giá trị trên, các chữ cái khác chỉ còn **một ẩn chưa biết**:

- `c['h'] = cnt[3] + cnt[8]` ⇒ `cnt[3] = c['h'] - cnt[8]`
- `c['f'] = cnt[4] + cnt[5]` ⇒ `cnt[5] = c['f'] - cnt[4]`
- `c['s'] = cnt[6] + cnt[7]` ⇒ `cnt[7] = c['s'] - cnt[6]`

Và cuối cùng:

- `c['o'] = cnt[0] + cnt[1] + cnt[2] + cnt[4]` ⇒ `cnt[1] = c['o'] - cnt[0] - cnt[2] - cnt[4]`
- `c['i'] = cnt[5] + cnt[6] + cnt[8] + cnt[9]` ⇒ `cnt[9] = c['i'] - cnt[5] - cnt[6] - cnt[8]`

Thứ tự tính: `0, 2, 4, 6, 8` → `3, 5, 7` → `1, 9`. Mỗi bước chỉ dùng giá trị đã biết. Hãy kiểm chứng các phương trình bằng cách sinh chuỗi thật.

In [10]:
rng = random.Random(6)
digits = [rng.randrange(10) for _ in range(40)]
s = make_valid(digits, rng)
c = Counter(s)
cnt = Counter(digits)
checks = {
    "z = cnt0": (c["z"], cnt[0]),
    "w = cnt2": (c["w"], cnt[2]),
    "u = cnt4": (c["u"], cnt[4]),
    "x = cnt6": (c["x"], cnt[6]),
    "g = cnt8": (c["g"], cnt[8]),
    "h = cnt3 + cnt8": (c["h"], cnt[3] + cnt[8]),
    "f = cnt4 + cnt5": (c["f"], cnt[4] + cnt[5]),
    "s = cnt6 + cnt7": (c["s"], cnt[6] + cnt[7]),
    "o = cnt0+1+2+4": (c["o"], cnt[0] + cnt[1] + cnt[2] + cnt[4]),
    "i = cnt5+6+8+9": (c["i"], cnt[5] + cnt[6] + cnt[8] + cnt[9]),
}
for name, (lhs, rhs) in checks.items():
    print(f"{name:<18}: {lhs} == {rhs} -> {lhs == rhs}")
    assert lhs == rhs

z = cnt0          : 4 == 4 -> True
w = cnt2          : 1 == 1 -> True
u = cnt4          : 5 == 5 -> True
x = cnt6          : 3 == 3 -> True
g = cnt8          : 3 == 3 -> True
h = cnt3 + cnt8   : 7 == 7 -> True
f = cnt4 + cnt5   : 10 == 10 -> True
s = cnt6 + cnt7   : 7 == 7 -> True
o = cnt0+1+2+4    : 16 == 16 -> True
i = cnt5+6+8+9    : 16 == 16 -> True


## 7. Các cách tiếp theo

### Cách 2 — Công thức bóc theo chữ cái độc quyền (`O(n)`)

**Giải thích từng đoạn:**

1. Đếm chữ cái bằng `Counter(s)`.
2. Tính `cnt[0], cnt[2], cnt[4], cnt[6], cnt[8]` trực tiếp từ `z, w, u, x, g`.
3. Tính `cnt[3], cnt[5], cnt[7]` bằng cách trừ phần chồng lấn.
4. Tính `cnt[1]`, `cnt[9]`.
5. Ghép `str(d) * cnt[d]` theo thứ tự `d = 0..9`.

In [11]:
def digits_formula(s):
    c = Counter(s)
    cnt = [0] * 10
    cnt[0] = c["z"]
    cnt[2] = c["w"]
    cnt[4] = c["u"]
    cnt[6] = c["x"]
    cnt[8] = c["g"]
    cnt[3] = c["h"] - cnt[8]
    cnt[5] = c["f"] - cnt[4]
    cnt[7] = c["s"] - cnt[6]
    cnt[1] = c["o"] - cnt[0] - cnt[2] - cnt[4]
    cnt[9] = c["i"] - cnt[5] - cnt[6] - cnt[8]
    return "".join(str(d) * cnt[d] for d in range(10))


for s, expected in examples:
    got = digits_formula(s)
    print(got, "| kỳ vọng", expected)
    assert got == expected

012 | kỳ vọng 012
45 | kỳ vọng 45


In [12]:
# Code nộp LeetCode — Cách 2
class Solution2:
    def originalDigits(self, s: str) -> str:
        c = Counter(s)
        cnt = [0] * 10
        cnt[0] = c["z"]
        cnt[2] = c["w"]
        cnt[4] = c["u"]
        cnt[6] = c["x"]
        cnt[8] = c["g"]
        cnt[3] = c["h"] - cnt[8]
        cnt[5] = c["f"] - cnt[4]
        cnt[7] = c["s"] - cnt[6]
        cnt[1] = c["o"] - cnt[0] - cnt[2] - cnt[4]
        cnt[9] = c["i"] - cnt[5] - cnt[6] - cnt[8]
        return "".join(str(d) * cnt[d] for d in range(10))


print(Solution2().originalDigits("fviefuro"))   # 45

45


**Độ phức tạp:** đếm `Counter(s)` là `O(n)`, phần còn lại `O(1)` ⇒ **Time `O(n)`**, Space `O(1)` (tối đa 15 chữ cái + mảng 10 phần tử).

### Cách 3 — Bóc theo bảng: "chữ cái khóa" và trừ trọn từ (table-driven)

**Ý tưởng:** thay vì nhớ 10 công thức, ta khai báo một **bảng** `ORDER = [(digit, key_letter), ...]` theo thứ tự bóc. Với mỗi cặp:

1. `times = counter[key_letter]` — số lần chữ số này xuất hiện (vì ở thời điểm bóc, `key_letter` chỉ còn thuộc về từ này).
2. Trừ **toàn bộ chữ cái của từ** `times` lần khỏi counter.

Thứ tự: `0(z), 2(w), 4(u), 6(x), 8(g), 3(h), 5(f), 7(s), 1(o), 9(i)`. Tại thời điểm xét `3` thì `h` chỉ còn ở `three` (vì `eight` đã bóc xong), v.v.

**Ưu điểm:** dễ mở rộng và dễ kiểm tra; nhược điểm: cần thứ tự bóc đúng.

In [13]:
ORDER = [(0, "z"), (2, "w"), (4, "u"), (6, "x"), (8, "g"),
         (3, "h"), (5, "f"), (7, "s"), (1, "o"), (9, "i")]


def digits_table(s):
    counter = Counter(s)
    cnt = [0] * 10
    for d, key in ORDER:
        times = counter[key]
        cnt[d] = times
        for ch in WORDS[d]:
            counter[ch] -= times
    return "".join(str(d) * cnt[d] for d in range(10))


for s, expected in examples:
    got = digits_table(s)
    print(got, "| kỳ vọng", expected)
    assert got == expected

012 | kỳ vọng 012
45 | kỳ vọng 45


In [14]:
# Code nộp LeetCode — Cách 3
class Solution3:
    def originalDigits(self, s: str) -> str:
        words = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
        order = [(0, "z"), (2, "w"), (4, "u"), (6, "x"), (8, "g"),
                 (3, "h"), (5, "f"), (7, "s"), (1, "o"), (9, "i")]
        counter = Counter(s)
        cnt = [0] * 10
        for d, key in order:
            times = counter[key]
            cnt[d] = times
            for ch in words[d]:
                counter[ch] -= times
        return "".join(str(d) * cnt[d] for d in range(10))


print(Solution3().originalDigits("owoztneoer"))   # 012

012


**Độ phức tạp:** `Counter(s)` `O(n)`; 10 lần bóc, mỗi lần `O(độ dài từ)` ⇒ **Time `O(n)`**, Space `O(1)`.

### Cách 4 — Hệ phương trình tuyến tính (khử Gauss với `Fraction`)

**Ý tưởng hoàn toàn khác:** coi bài toán là giải hệ `A x = b`:

- `A` là ma trận `15 x 10`: `A[l][d]` = số lần chữ cái `l` có trong từ của chữ số `d`.
- `b[l]` = số lần chữ cái `l` có trong `s`.
- `x[d]` = `cnt[d]` cần tìm.

Nếu các cột của `A` độc lập tuyến tính (hạng bằng 10) thì nghiệm **duy nhất** — điều này giải thích vì sao đáp án luôn xác định. Ta khử Gauss–Jordan bằng `Fraction` (phân số chính xác) để tránh sai số số thực.

**Giải thích từng đoạn:**

1. Dựng ma trận mở rộng `[A | b]`.
2. Với mỗi cột `d`, tìm dòng có phần tử khác `0` (pivot), đổi lên, chia dòng cho pivot, rồi khử cột đó ở các dòng khác.
3. Sau 10 pivot, nghiệm `x[d]` nằm ở cột cuối của dòng pivot tương ứng.

In [15]:
LETTERS = sorted(set("".join(WORDS)))     # 15 chữ cái
A = [[WORDS[d].count(l) for d in range(10)] for l in LETTERS]


def rank_of(matrix):
    m = [[Fraction(x) for x in row] for row in matrix]
    rank, rows, cols = 0, len(m), len(m[0])
    for col in range(cols):
        piv = next((r for r in range(rank, rows) if m[r][col] != 0), None)
        if piv is None:
            continue
        m[rank], m[piv] = m[piv], m[rank]
        m[rank] = [x / m[rank][col] for x in m[rank]]
        for r in range(rows):
            if r != rank and m[r][col] != 0:
                f = m[r][col]
                m[r] = [a - f * b for a, b in zip(m[r], m[rank])]
        rank += 1
    return rank


print("ma trận A (15 x 10), mỗi hàng là một chữ cái:")
print("       " + " ".join(f"{d:>2}" for d in range(10)))
for l, row in zip(LETTERS, A):
    print(f"  {l}:   " + " ".join(f"{v:>2}" for v in row))
print("hạng của A =", rank_of(A), "(=10 -> nghiệm duy nhất)")

ma trận A (15 x 10), mỗi hàng là một chữ cái:
        0  1  2  3  4  5  6  7  8  9
  e:    1  1  0  2  0  1  0  2  1  1
  f:    0  0  0  0  1  1  0  0  0  0
  g:    0  0  0  0  0  0  0  0  1  0
  h:    0  0  0  1  0  0  0  0  1  0
  i:    0  0  0  0  0  1  1  0  1  1
  n:    0  1  0  0  0  0  0  1  0  2
  o:    1  1  1  0  1  0  0  0  0  0
  r:    1  0  0  1  1  0  0  0  0  0
  s:    0  0  0  0  0  0  1  1  0  0
  t:    0  0  1  1  0  0  0  0  1  0
  u:    0  0  0  0  1  0  0  0  0  0
  v:    0  0  0  0  0  1  0  1  0  0
  w:    0  0  1  0  0  0  0  0  0  0
  x:    0  0  0  0  0  0  1  0  0  0
  z:    1  0  0  0  0  0  0  0  0  0
hạng của A = 10 (=10 -> nghiệm duy nhất)


In [16]:
def digits_gauss(s):
    c = Counter(s)
    aug = [[Fraction(A[i][d]) for d in range(10)] + [Fraction(c[LETTERS[i]])] for i in range(len(LETTERS))]
    rank = 0
    pivot_row_of_col = {}
    for col in range(10):
        piv = next((r for r in range(rank, len(aug)) if aug[r][col] != 0), None)
        aug[rank], aug[piv] = aug[piv], aug[rank]
        p = aug[rank][col]
        aug[rank] = [x / p for x in aug[rank]]
        for r in range(len(aug)):
            if r != rank and aug[r][col] != 0:
                f = aug[r][col]
                aug[r] = [a - f * b for a, b in zip(aug[r], aug[rank])]
        pivot_row_of_col[col] = rank
        rank += 1
    cnt = [aug[pivot_row_of_col[d]][10] for d in range(10)]
    assert all(x.denominator == 1 and x >= 0 for x in cnt), cnt      # đề đảm bảo hợp lệ
    return "".join(str(d) * int(cnt[d]) for d in range(10))


for s, expected in examples:
    got = digits_gauss(s)
    print(got, "| kỳ vọng", expected)
    assert got == expected

012 | kỳ vọng 012
45 | kỳ vọng 45


Cách 4 không phải để nộp (chậm hơn và dài hơn) mà để **hiểu bản chất**: nghiệm duy nhất, và các công thức ở Cách 2 chính là kết quả của việc khử Gauss khi ta chọn đúng thứ tự biến.

**Độ phức tạp:** đếm `O(n)`; khử Gauss trên ma trận cố định `15 x 11` là `O(1)` ⇒ **Time `O(n)`**, Space `O(1)`.

**Bốn cách tóm tắt:** (1) quay lui thử mọi cách chia; (2) 10 công thức từ chữ cái độc quyền; (3) bảng thứ tự + trừ trọn từ; (4) giải hệ tuyến tính. Cách 2 và 3 cùng ý nhưng khác cách cài đặt; Cách 1 và 4 là hai góc nhìn khác hẳn (tìm kiếm vs đại số).

## 8. TRACE từng bước

Ta trace **Cách 3** trên chuỗi tự dựng gồm các chữ số `[9, 8, 8, 3, 1, 0, 7]` đã xáo trộn. Với seed cố định ta in chuỗi và từng bước bóc.

Bảng trạng thái (bạn nên tự dự đoán trước):

| bước | chữ số | chữ cái khóa | `counter[key]` lúc bóc | thao tác |
|---|---|---|---|---|
| 1 | 0 | z | 1 | trừ `zero` 1 lần |
| 2 | 2 | w | 0 | không có `two` |
| 3 | 4 | u | 0 | không có `four` |
| 4 | 6 | x | 0 | không có `six` |
| 5 | 8 | g | 2 | trừ `eight` 2 lần |
| 6 | 3 | h | ? | ? |
| ... | | | | |

In [17]:
def trace_table(s):
    counter = Counter(s)
    print("s =", s, "| đếm ban đầu:", dict(sorted(counter.items())))
    cnt = [0] * 10
    for step, (d, key) in enumerate(ORDER, 1):
        times = counter[key]
        cnt[d] = times
        before = dict(sorted((k, v) for k, v in counter.items() if v))
        for ch in WORDS[d]:
            counter[ch] -= times
        after = {k: v for k, v in sorted(counter.items()) if v}
        print(f"bước {step:>2}: chữ số {d} ({WORDS[d]:<5}) khóa {key!r} -> xuất hiện {times} lần | còn lại {after}")
    result = "".join(str(d) * cnt[d] for d in range(10))
    print("kết quả:", repr(result), "| mọi chữ cái đã về 0?", all(v == 0 for v in counter.values()))
    return result


digits = [9, 8, 8, 3, 1, 0, 7]
s = make_valid(digits, random.Random(8))
assert trace_table(s) == "".join(map(str, sorted(digits)))

s = ioeieeterteenoehtnvsrnenizgehgh | đếm ban đầu: {'e': 9, 'g': 2, 'h': 3, 'i': 3, 'n': 4, 'o': 2, 'r': 2, 's': 1, 't': 3, 'v': 1, 'z': 1}
bước  1: chữ số 0 (zero ) khóa 'z' -> xuất hiện 1 lần | còn lại {'e': 8, 'g': 2, 'h': 3, 'i': 3, 'n': 4, 'o': 1, 'r': 1, 's': 1, 't': 3, 'v': 1}
bước  2: chữ số 2 (two  ) khóa 'w' -> xuất hiện 0 lần | còn lại {'e': 8, 'g': 2, 'h': 3, 'i': 3, 'n': 4, 'o': 1, 'r': 1, 's': 1, 't': 3, 'v': 1}
bước  3: chữ số 4 (four ) khóa 'u' -> xuất hiện 0 lần | còn lại {'e': 8, 'g': 2, 'h': 3, 'i': 3, 'n': 4, 'o': 1, 'r': 1, 's': 1, 't': 3, 'v': 1}
bước  4: chữ số 6 (six  ) khóa 'x' -> xuất hiện 0 lần | còn lại {'e': 8, 'g': 2, 'h': 3, 'i': 3, 'n': 4, 'o': 1, 'r': 1, 's': 1, 't': 3, 'v': 1}
bước  5: chữ số 8 (eight) khóa 'g' -> xuất hiện 2 lần | còn lại {'e': 6, 'h': 1, 'i': 1, 'n': 4, 'o': 1, 'r': 1, 's': 1, 't': 1, 'v': 1}
bước  6: chữ số 3 (three) khóa 'h' -> xuất hiện 1 lần | còn lại {'e': 4, 'i': 1, 'n': 4, 'o': 1, 's': 1, 'v': 1}
bước  7: chữ số 5 (five ) khóa

Đọc output: ở bước `8` (`g`) ta bóc 2 từ `eight`; ở bước `3` (`h`), `h` chỉ còn của `three` vì `eight` đã biến mất khỏi counter; bước `1` (`o`) chỉ còn của `one` vì `zero`, `two`, `four` đã bóc. Cuối cùng mọi chữ cái về `0` — dấu hiệu nghiệm đúng.

Trace **Cách 2** trên `s = "fviefuro"`: ta in từng công thức với giá trị cụ thể.

In [18]:
def trace_formula(s):
    c = Counter(s)
    print("đếm:", dict(sorted(c.items())))
    cnt = [0] * 10
    steps = [
        (0, "c['z']", lambda: c["z"]),
        (2, "c['w']", lambda: c["w"]),
        (4, "c['u']", lambda: c["u"]),
        (6, "c['x']", lambda: c["x"]),
        (8, "c['g']", lambda: c["g"]),
        (3, "c['h'] - cnt[8]", lambda: c["h"] - cnt[8]),
        (5, "c['f'] - cnt[4]", lambda: c["f"] - cnt[4]),
        (7, "c['s'] - cnt[6]", lambda: c["s"] - cnt[6]),
        (1, "c['o'] - cnt[0] - cnt[2] - cnt[4]", lambda: c["o"] - cnt[0] - cnt[2] - cnt[4]),
        (9, "c['i'] - cnt[5] - cnt[6] - cnt[8]", lambda: c["i"] - cnt[5] - cnt[6] - cnt[8]),
    ]
    for d, text, fn in steps:
        cnt[d] = fn()
        print(f"cnt[{d}] = {text:<36} = {cnt[d]}")
    print("cnt =", cnt)
    return "".join(str(d) * cnt[d] for d in range(10))


assert trace_formula("fviefuro") == "45"

đếm: {'e': 1, 'f': 2, 'i': 1, 'o': 1, 'r': 1, 'u': 1, 'v': 1}
cnt[0] = c['z']                               = 0
cnt[2] = c['w']                               = 0
cnt[4] = c['u']                               = 1
cnt[6] = c['x']                               = 0
cnt[8] = c['g']                               = 0
cnt[3] = c['h'] - cnt[8]                      = 0
cnt[5] = c['f'] - cnt[4]                      = 1
cnt[7] = c['s'] - cnt[6]                      = 0
cnt[1] = c['o'] - cnt[0] - cnt[2] - cnt[4]    = 0
cnt[9] = c['i'] - cnt[5] - cnt[6] - cnt[8]    = 0
cnt = [0, 0, 0, 0, 1, 1, 0, 0, 0, 0]


Cuối cùng trace **Cách 4** trên ví dụ nhỏ: in ma trận mở rộng sau mỗi bước chọn pivot (chỉ in cột cần thiết) để thấy nghiệm hiện ra.

In [19]:
def trace_gauss(s, show_steps=3):
    c = Counter(s)
    aug = [[Fraction(A[i][d]) for d in range(10)] + [Fraction(c[LETTERS[i]])] for i in range(len(LETTERS))]
    rank = 0
    for col in range(10):
        piv = next(r for r in range(rank, len(aug)) if aug[r][col] != 0)
        aug[rank], aug[piv] = aug[piv], aug[rank]
        p = aug[rank][col]
        aug[rank] = [x / p for x in aug[rank]]
        for r in range(len(aug)):
            if r != rank and aug[r][col] != 0:
                f = aug[r][col]
                aug[r] = [a - f * b for a, b in zip(aug[r], aug[rank])]
        if col < show_steps:
            print(f"-- sau khi chọn pivot cột {col}: (dòng pivot hiện ở vị trí {rank})")
            for r in range(len(aug)):
                print("   ", " ".join(f"{str(x):>3}" for x in aug[r][:10]), "|", aug[r][10])
        rank += 1
    return [int(aug[d][10]) for d in range(10)]


print(trace_gauss("fviefuro", show_steps=1))

-- sau khi chọn pivot cột 0: (dòng pivot hiện ở vị trí 0)
      1   1   0   2   0   1   0   2   1   1 | 1
      0   0   0   0   1   1   0   0   0   0 | 2
      0   0   0   0   0   0   0   0   1   0 | 0
      0   0   0   1   0   0   0   0   1   0 | 0
      0   0   0   0   0   1   1   0   1   1 | 1
      0   1   0   0   0   0   0   1   0   2 | 0
      0   0   1  -2   1  -1   0  -2  -1  -1 | 0
      0  -1   0  -1   1  -1   0  -2  -1  -1 | 0
      0   0   0   0   0   0   1   1   0   0 | 0
      0   0   1   1   0   0   0   0   1   0 | 0
      0   0   0   0   1   0   0   0   0   0 | 1
      0   0   0   0   0   1   0   1   0   0 | 1
      0   0   1   0   0   0   0   0   0   0 | 0
      0   0   0   0   0   0   1   0   0   0 | 0
      0  -1   0  -2   0  -1   0  -2  -1  -1 | -1
[0, 0, 0, 0, 1, 1, 0, 0, 0, 0]


## 9. Code nộp LeetCode

Bản khuyến nghị: **Cách 2** (công thức tường minh) hoặc Cách 3 (bảng). Ta nộp Cách 2 vì ngắn và nhanh. Lưu ý: cần `from collections import Counter` khi chạy ngoài LeetCode (trên LeetCode `Counter` đã sẵn).

In [20]:
class Solution:
    def originalDigits(self, s: str) -> str:
        c = Counter(s)
        cnt = [0] * 10
        # Bước 1: chữ cái độc quyền
        cnt[0] = c["z"]               # zero
        cnt[2] = c["w"]               # two
        cnt[4] = c["u"]               # four
        cnt[6] = c["x"]               # six
        cnt[8] = c["g"]               # eight
        # Bước 2: trừ phần chồng lấn đã biết
        cnt[3] = c["h"] - cnt[8]      # three (h còn ở eight)
        cnt[5] = c["f"] - cnt[4]      # five  (f còn ở four)
        cnt[7] = c["s"] - cnt[6]      # seven (s còn ở six)
        # Bước 3: hai chữ số cuối
        cnt[1] = c["o"] - cnt[0] - cnt[2] - cnt[4]    # one  (o ở zero, two, four)
        cnt[9] = c["i"] - cnt[5] - cnt[6] - cnt[8]    # nine (i ở five, six, eight)
        return "".join(str(d) * cnt[d] for d in range(10))      # tăng dần theo d


for s, expected in examples:
    got = Solution().originalDigits(s)
    print(got, expected)
    assert got == expected

012 012
45 45


## 10. Vì sao đúng

**Mô hình.** Chuỗi hợp lệ `s` là một hoán vị các chữ cái của đa tập `{ WORDS[d] lặp cnt[d] lần : d = 0..9 }` với `cnt[d] >= 0` nào đó. Với mỗi chữ cái `x`: `c[x] = Σ_d A[x][d] * cnt[d]` (`A[x][d]` là số lần `x` trong `WORDS[d]`).

**Mỗi công thức là một hệ quả trực tiếp của phương trình tương ứng:**

- `z` chỉ có trong `zero` với số lần `1` ⇒ `c[z] = cnt[0]`. Tương tự `w → 2`, `u → 4`, `x → 6`, `g → 8`.
- `h` có trong `three` (1 lần) và `eight` (1 lần) ⇒ `c[h] = cnt[3] + cnt[8]` ⇒ `cnt[3] = c[h] - cnt[8]` (đã biết `cnt[8]`).
- `f`: `four`, `five` ⇒ `cnt[5] = c[f] - cnt[4]`. `s`: `six`, `seven` ⇒ `cnt[7] = c[s] - cnt[6]`.
- `o`: `zero`, `one`, `two`, `four` (mỗi từ 1 lần `o`) ⇒ `cnt[1] = c[o] - cnt[0] - cnt[2] - cnt[4]`.
- `i`: `five`, `six`, `eight`, `nine` ⇒ `cnt[9] = c[i] - cnt[5] - cnt[6] - cnt[8]`.

Đủ 10 chữ số, mỗi chữ số được xác định **duy nhất** bởi một phương trình mà các ẩn còn lại đã biết (hệ tam giác). Vậy nghiệm xác định duy nhất (khớp với việc hạng của `A` bằng 10).

**Vì sao ghép theo `d = 0..9` cho thứ tự tăng dần?** Vòng `for d in range(10)` nối `str(d) * cnt[d]` theo `d` tăng dần, đúng yêu cầu "tăng dần".

**Đảm bảo hợp lệ.** Vì `s` hợp lệ nên có `cnt` thật thỏa mọi phương trình; công thức tìm lại đúng `cnt` đó (do nghiệm duy nhất). Nếu `s` không hợp lệ, một số `cnt[d]` có thể âm hoặc còn thừa chữ cái — ta sẽ kiểm tra điều này ở biến thể 3.

**Kiểm chứng bằng code:** các test ở mục 12 dựng `s` từ một danh sách chữ số thật rồi xáo trộn; nếu công thức đúng thì kết quả luôn là danh sách đó đã sắp xếp.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Quay lui | mũ (không dùng cho input lớn) | `O(1)` |
| 2. Công thức | `O(n)` | `O(1)` |
| 3. Bảng + trừ trọn từ | `O(n)` | `O(1)` |
| 4. Khử Gauss | `O(n)` (ma trận cố định `15 x 11`) | `O(1)` |

Phần `O(n)` chỉ là đếm chữ cái; phần còn lại là hằng số. Với `n = 10^5` chạy trong vài mili-giây. Hãy đo.

In [21]:
rng = random.Random(11)
digits = [rng.randrange(10) for _ in range(20000)]      # ~ 90.000 chữ cái, gần giới hạn 10^5
s = make_valid(digits, rng)
print("độ dài s =", len(s))
expected = "".join(map(str, sorted(digits)))
for name, f in [("Cách 2", digits_formula), ("Cách 3", digits_table), ("Cách 4", digits_gauss)]:
    t = time.perf_counter()
    r = f(s)
    print(f"{name}: {time.perf_counter() - t:.4f} giây, đúng = {r == expected}")

độ dài s = 79951
Cách 2: 0.0024 giây, đúng = True
Cách 3: 0.0025 giây, đúng = True
Cách 4: 0.0031 giây, đúng = True


## 12. Bộ test

Oracle độc lập: ta **sinh** `s` từ một danh sách chữ số đã biết (ghép các từ rồi `shuffle` với seed), nên đáp án đúng chính là danh sách đó sắp xếp — không phụ thuộc vào bất kỳ thuật toán nào. Với chuỗi ngắn ta thêm đối chiếu với quay lui.

In [22]:
solutions = {
    "formula": digits_formula,
    "table": digits_table,
    "gauss": digits_gauss,
    "LeetCode Solution": lambda s: Solution().originalDigits(s),
    "Solution2": lambda s: Solution2().originalDigits(s),
    "Solution3": lambda s: Solution3().originalDigits(s),
}

# 12.1. Ví dụ đề
for s, expected in examples:
    for name, f in solutions.items():
        assert f(s) == expected, (name, s)
    assert digits_brute(s) == expected
print("12.1 OK")

12.1 OK


In [23]:
# 12.2. Edge case: từng chữ số một mình, mỗi chữ số lặp nhiều lần, đủ 0..9
rng = random.Random(0)
for d in range(10):
    s = make_valid([d], rng)
    for name, f in solutions.items():
        assert f(s) == str(d), (name, d, s)
    assert digits_brute(s) == str(d)

for d in range(10):                                   # lặp một chữ số 7 lần
    s = make_valid([d] * 7, rng)
    for name, f in solutions.items():
        assert f(s) == str(d) * 7, (name, d)

s = make_valid(list(range(10)), rng)                  # đủ 0..9
for name, f in solutions.items():
    assert f(s) == "0123456789", name
assert digits_brute(s) == "0123456789"

# các chữ số cố ý gây chồng lấn: 'nine', 'one', 'seven' (chung n, e)
s = make_valid([9, 1, 7, 9], rng)
for name, f in solutions.items():
    assert f(s) == "1799", name
print("12.2 OK")

12.2 OK


In [24]:
# 12.3. Ngẫu nhiên có seed: sinh chữ số thật -> xáo trộn -> khôi phục
rng = random.Random(423)
for _ in range(3000):
    digits = [rng.randrange(10) for _ in range(rng.randint(1, 30))]
    s = make_valid(digits, rng)
    expected = "".join(map(str, sorted(digits)))
    for name, f in solutions.items():
        got = f(s)
        assert got == expected, (name, digits, s, got)
print("12.3 OK: 3000 test ngẫu nhiên khớp dữ liệu sinh ở cả 6 cách")

12.3 OK: 3000 test ngẫu nhiên khớp dữ liệu sinh ở cả 6 cách


In [25]:
# 12.4. Đối chiếu với quay lui trên chuỗi ngắn (oracle tìm kiếm độc lập)
rng = random.Random(4230)
for _ in range(60):
    digits = [rng.randrange(10) for _ in range(rng.randint(1, 5))]
    s = make_valid(digits, rng)
    assert digits_brute(s) == digits_formula(s) == "".join(map(str, sorted(digits)))
print("12.4 OK: 60 test quay lui")

12.4 OK: 60 test quay lui


In [26]:
# 12.5. Gần giới hạn: |s| gần 10^5, các cách nhanh phải khớp
rng = random.Random(77)
for _ in range(5):
    digits = [rng.randrange(10) for _ in range(rng.randint(15000, 20000))]
    s = make_valid(digits, rng)
    assert len(s) <= 10**5 + 4
    expected = "".join(map(str, sorted(digits)))
    assert Solution().originalDigits(s) == digits_table(s) == digits_gauss(s) == expected
print("12.5 OK")

12.5 OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng giá trị chưa biết (tính sai thứ tự)

Công thức `cnt[1] = c['o'] - cnt[0] - cnt[2] - cnt[4]` chỉ đúng khi `cnt[0], cnt[2], cnt[4]` **đã được tính**. Nếu tính `cnt[1]` trước, các phần tử đó còn là `0` ⇒ `cnt[1]` quá lớn.

In [27]:
def wrong_order(s):
    c = Counter(s)
    cnt = [0] * 10
    cnt[1] = c["o"] - cnt[0] - cnt[2] - cnt[4]     # SAI: cnt[0], cnt[2], cnt[4] chưa được tính (đang bằng 0)
    cnt[0] = c["z"]
    cnt[2] = c["w"]
    cnt[4] = c["u"]
    cnt[6] = c["x"]
    cnt[8] = c["g"]
    cnt[3] = c["h"] - cnt[8]
    cnt[5] = c["f"] - cnt[4]
    cnt[7] = c["s"] - cnt[6]
    cnt[9] = c["i"] - cnt[5] - cnt[6] - cnt[8]
    return "".join(str(d) * cnt[d] for d in range(10))


print("sai  :", wrong_order("owoztneoer"))
print("đúng :", digits_formula("owoztneoer"))

sai  : 01112
đúng : 012


**Vì sao sai?** `owoztneoer` có `o = 3` nhưng `o` đến từ `zero`, `one`, `two`; nếu chưa trừ `cnt[0], cnt[2]` thì `cnt[1] = 3` thay vì `1`. Sửa: tính các chữ số có chữ cái độc quyền trước, rồi mới tới chữ số phụ thuộc.

### Lỗi 2 — Quên trừ phần chồng lấn (`h` cũng có ở `eight`)

In [28]:
def wrong_no_subtract(s):
    c = Counter(s)
    cnt = [0] * 10
    cnt[0], cnt[2], cnt[4], cnt[6], cnt[8] = c["z"], c["w"], c["u"], c["x"], c["g"]
    cnt[3] = c["h"]                                  # SAI: h cũng có trong eight
    cnt[5] = c["f"] - cnt[4]
    cnt[7] = c["s"] - cnt[6]
    cnt[1] = c["o"] - cnt[0] - cnt[2] - cnt[4]
    cnt[9] = c["i"] - cnt[5] - cnt[6] - cnt[8]
    return "".join(str(d) * cnt[d] for d in range(10))


rng = random.Random(2)
s = make_valid([8, 8, 3], rng)                       # eight eight three
print("s =", s)
print("sai  :", wrong_no_subtract(s))
print("đúng :", digits_formula(s))

s = thirtghtgehieee
sai  : 33388
đúng : 388


**Vì sao sai?** `h` xuất hiện ở cả `three` lẫn `eight`. Có hai chữ `eight` và một `three` ⇒ `c['h'] = 3`, nhưng `cnt[3]` thật là `3 - 2 = 1`.

### Lỗi 3 — Ghép kết quả theo thứ tự "tìm ra" thay vì `0..9`

In [29]:
def wrong_output_order(s):
    c = Counter(s)
    cnt = [0] * 10
    cnt[0], cnt[2], cnt[4], cnt[6], cnt[8] = c["z"], c["w"], c["u"], c["x"], c["g"]
    cnt[3] = c["h"] - cnt[8]
    cnt[5] = c["f"] - cnt[4]
    cnt[7] = c["s"] - cnt[6]
    cnt[1] = c["o"] - cnt[0] - cnt[2] - cnt[4]
    cnt[9] = c["i"] - cnt[5] - cnt[6] - cnt[8]
    found_order = [0, 2, 4, 6, 8, 3, 5, 7, 1, 9]        # SAI: ghép theo thứ tự tính toán
    return "".join(str(d) * cnt[d] for d in found_order)


print("sai  :", wrong_output_order("owoztneoer"))
print("đúng :", digits_formula("owoztneoer"))

sai  : 021
đúng : 012


**Vì sao sai?** Đề yêu cầu **tăng dần**; ta tính theo thứ tự `0,2,4,6,8,3,5,7,1,9` nhưng phải **in** theo `0..9`. Tách "thứ tự tính" khỏi "thứ tự in".

### Lỗi 4 — Greedy cắt từ theo thứ tự `0..9`

Ý tưởng "với mỗi chữ số từ 0 đến 9, cứ cắt từ nếu còn đủ chữ cái" nghe hợp lý nhưng **cướp** chữ cái của từ khác. Ví dụ `one` có thể lấy `o` của `two` và `n, e` của `nine`.

In [30]:
def wrong_greedy(s):
    c = Counter(s)
    result = []
    for d, w in enumerate(WORDS):
        need = Counter(w)
        while all(c[ch] >= k for ch, k in need.items()):     # SAI: cắt tham lam
            for ch, k in need.items():
                c[ch] -= k
            result.append(str(d))
    return "".join(result)


rng = random.Random(3)
found = None
for _ in range(5000):
    digits = [rng.randrange(10) for _ in range(rng.randint(1, 4))]
    s = make_valid(digits, rng)
    if wrong_greedy(s) != "".join(map(str, sorted(digits))):
        found = (digits, s)
        break
print("phản ví dụ:", found)
print("sai  :", wrong_greedy(found[1]))
print("đúng :", "".join(map(str, sorted(found[0]))))

phản ví dụ: ([0, 4, 7], 'osofzrernvuee')
sai  : 01
đúng : 047


**Vì sao sai?** Greedy theo thứ tự chữ số không có "chữ cái khóa" nên cắt nhầm; kết quả hoặc dư chữ cái hoặc ra chữ số khác. Chỉ khi bóc theo thứ tự **độc quyền → phụ thuộc** ta mới chắc chắn mỗi lần bóc là đúng.

**Bài học chung:** khi nhiều đối tượng dùng chung "tài nguyên" (chữ cái), hãy tìm thứ tự xử lý sao cho mỗi bước chỉ có **một lựa chọn bắt buộc** — đó là tam giác hóa.

## 14. Mẹo & nhận dạng pattern

- **"Xáo trộn các từ rồi hỏi tập từ gốc"** ⇒ đếm tần suất chữ cái, tìm chữ cái (hoặc tổ hợp) **độc quyền** để xác định số lần mỗi từ.
- **Hệ thống "tam giác"**: bóc từng lớp sao cho mỗi bước chỉ còn một ẩn; đây là dạng khử Gauss "bằng mắt".
- Nếu không thấy ngay chữ cái độc quyền, hãy lập bảng "chữ cái nào thuộc những từ nào" (mục 3.4) hoặc dùng đại số (Cách 4).
- Kiểm tra bằng **sinh dữ liệu ngược**: tạo đáp án trước, rồi tạo đề từ đáp án (thường dùng khi bài có nghiệm duy nhất).
- `str(d) * cnt[d]` là thủ thuật gọn để lặp chữ số đúng số lần (kể cả `0` lần).
- Ở bài này mọi thứ là `O(1)` sau khi đếm; chi phí chính là một lượt `Counter(s)`.

## 15. Biến thể

1. **Trả về số lần xuất hiện của từng chữ số** (mảng `cnt`) thay vì chuỗi: bỏ bước ghép.
2. **Có chữ in hoa lẫn thường / có khoảng trắng:** chuẩn hóa `s.lower()` và bỏ ký tự thừa trước khi đếm.
3. **Kiểm tra `s` có hợp lệ không (không đảm bảo):** tính `cnt` rồi kiểm tra không có `cnt[d] < 0`, và tổng chữ cái của các từ khớp `Counter(s)`.
4. **Ngôn ngữ khác (tiếng Việt không dấu: `khong, mot, hai, ba, bon, nam, sau, bay, tam, chin`):** cần tìm lại chữ cái độc quyền (hoặc khử Gauss — Cách 4 tổng quát hơn).
5. **Từ điển tùy ý:** thay `WORDS` và để Cách 4 tự tìm nghiệm (cần hạng ma trận đủ).

In [31]:
# Biến thể 3: kiểm tra s có phải là hoán vị hợp lệ của các từ chữ số hay không
def is_valid_digit_string(s):
    c = Counter(s)
    cnt = [0] * 10
    cnt[0], cnt[2], cnt[4], cnt[6], cnt[8] = c["z"], c["w"], c["u"], c["x"], c["g"]
    cnt[3] = c["h"] - cnt[8]
    cnt[5] = c["f"] - cnt[4]
    cnt[7] = c["s"] - cnt[6]
    cnt[1] = c["o"] - cnt[0] - cnt[2] - cnt[4]
    cnt[9] = c["i"] - cnt[5] - cnt[6] - cnt[8]
    if any(x < 0 for x in cnt):
        return False
    rebuilt = Counter("".join(WORDS[d] * cnt[d] for d in range(10)))
    return rebuilt == c


print(is_valid_digit_string("owoztneoer"), is_valid_digit_string("owoztneoea"), is_valid_digit_string("abc"))

True False False


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | Thử mọi `cnt[d]` | mũ | `O(1)` | Oracle cho input rất nhỏ |
| 2. Công thức | 10 phương trình tam giác viết tay | `O(n)` | `O(1)` | **Khuyên dùng khi thi** |
| 3. Bảng + trừ trọn từ | `ORDER` (chữ số, chữ cái khóa) | `O(n)` | `O(1)` | Dễ mở rộng, dễ kiểm tra |
| 4. Khử Gauss | `A x = b` với `Fraction` | `O(n)` | `O(1)` | Hiểu nghiệm duy nhất; từ điển tùy ý |

Cả bốn cho cùng đáp án trên mọi test ở mục 12.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `"7" * 3`; (b) `Counter("eight")["e"]` và `Counter("eight")["z"]`; (c) `"".join(str(d) * c for d, c in enumerate([0, 2, 1]))`; (d) chữ cái nào có trong `nine` nhưng xuất hiện **hai lần**?

**Bài 2 (tính tay).** Với `s = "seenvxegihtis"` (các chữ cái của `six`, `seven`, `eight` đã xáo trộn): đếm chữ cái, tính `cnt[6]`, `cnt[8]`, `cnt[7]` bằng công thức và suy ra đáp án.

**Bài 3 (điền TODO).** Hoàn thành `original_digits_todo(s)` theo Cách 3 (bảng).

In [32]:
def original_digits_todo(s):
    # TODO 1: counter = Counter(s); cnt = [0] * 10
    # TODO 2: for d, key in ORDER: times = counter[key]; cnt[d] = times
    # TODO 3: trừ từng chữ cái của WORDS[d] đi times lần khỏi counter
    # TODO 4: trả "".join(str(d) * cnt[d] for d in range(10))
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `original_digits_counts(s)` trả về **danh sách `cnt` độ dài 10** (số lần mỗi chữ số) bằng Cách 2 **không nhìn lại** mục 7. Kiểm tra: `"owoztneoer"` → `[1, 1, 1, 0, 0, 0, 0, 0, 0, 0]`.

**Bài 5 (biến thể / mini challenge).** Viết `can_form(s)` trả về `True/False` nếu `s` là hoán vị hợp lệ của các từ chữ số (không đảm bảo hợp lệ). Kiểm tra: `"owoztneoer"` → `True`; `"owoztneoea"` → `False`; `"zz"` → `False`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `'777'`; `2` và `0`; `'1' * 0 + ...` — nhớ `enumerate` cho `(0,0), (1,2), (2,1)` ⇒ `'' + '11' + '2'`; chữ `n` (`nine` có hai chữ `n`).
- **Mức 2 (vừa):** Bài 2: `"seenvxegihtis"` có `x = 1` ⇒ `cnt[6] = 1`; `g = 1` ⇒ `cnt[8] = 1`; `s = 2` ⇒ `cnt[7] = s - cnt[6] = 1`. Đáp án `"678"`. Bài 3: sao chép khung vòng `for d, key in ORDER` ở mục 7.
- **Mức 3 (sâu):** Bài 4: copy công thức rồi `return cnt`. Bài 5: tính `cnt`, kiểm tra không âm, rồi dựng lại chuỗi `"".join(WORDS[d] * cnt[d] ...)` và so sánh `Counter` với `Counter(s)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert "7" * 3 == "777"
assert (Counter("eight")["e"], Counter("eight")["z"]) == (1, 0)
assert "".join(str(d) * c for d, c in enumerate([0, 2, 1])) == "112"
assert Counter("nine")["n"] == 2

# Bài 2
s2 = "seenvxegihtis"
assert Counter(s2) == Counter("sixseveneight")
c2 = Counter(s2)
assert (c2["x"], c2["g"], c2["s"] - c2["x"]) == (1, 1, 1)
assert digits_formula(s2) == "678"


# Bài 3
def original_digits_todo_ans(s):
    counter = Counter(s)
    cnt = [0] * 10
    for d, key in ORDER:
        times = counter[key]
        cnt[d] = times
        for ch in WORDS[d]:
            counter[ch] -= times
    return "".join(str(d) * cnt[d] for d in range(10))


for s, expected in examples:
    assert original_digits_todo_ans(s) == expected


# Bài 4
def original_digits_counts(s):
    c = Counter(s)
    cnt = [0] * 10
    cnt[0], cnt[2], cnt[4], cnt[6], cnt[8] = c["z"], c["w"], c["u"], c["x"], c["g"]
    cnt[3] = c["h"] - cnt[8]
    cnt[5] = c["f"] - cnt[4]
    cnt[7] = c["s"] - cnt[6]
    cnt[1] = c["o"] - cnt[0] - cnt[2] - cnt[4]
    cnt[9] = c["i"] - cnt[5] - cnt[6] - cnt[8]
    return cnt


assert original_digits_counts("owoztneoer") == [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]


# Bài 5
def can_form(s):
    cnt = original_digits_counts(s)
    if any(x < 0 for x in cnt):
        return False
    return Counter("".join(WORDS[d] * cnt[d] for d in range(10))) == Counter(s)


assert can_form("owoztneoer") is True
assert can_form("owoztneoea") is False
assert can_form("zz") is False
rng = random.Random(19)
for _ in range(300):
    digits = [rng.randrange(10) for _ in range(rng.randint(1, 8))]
    s = make_valid(digits, rng)
    assert can_form(s)
    assert original_digits_counts(s) == [digits.count(d) for d in range(10)]
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (10 công thức từ chữ cái độc quyền)

```python
class Solution:
    def originalDigits(self, s: str) -> str:
        c = Counter(s)
        cnt = [0] * 10
        cnt[0], cnt[2], cnt[4], cnt[6], cnt[8] = c["z"], c["w"], c["u"], c["x"], c["g"]
        cnt[3] = c["h"] - cnt[8]
        cnt[5] = c["f"] - cnt[4]
        cnt[7] = c["s"] - cnt[6]
        cnt[1] = c["o"] - cnt[0] - cnt[2] - cnt[4]
        cnt[9] = c["i"] - cnt[5] - cnt[6] - cnt[8]
        return "".join(str(d) * cnt[d] for d in range(10))
```

Nhớ nhanh: **z-w-u-x-g** (0-2-4-6-8) → **h-f-s** (3-5-7) → **o-i** (1-9).

**Checklist**

- [ ] Đã tính các chữ số có chữ cái **độc quyền** (0,2,4,6,8) **trước**?
- [ ] Đã trừ phần chồng lấn: `h` trừ `8`; `f` trừ `4`; `s` trừ `6`?
- [ ] `cnt[1]` trừ đủ `0, 2, 4` và `cnt[9]` trừ đủ `5, 6, 8`?
- [ ] Ghép kết quả theo `d = 0..9` (tăng dần), không theo thứ tự tính?
- [ ] Đã import `Counter` (khi chạy ngoài LeetCode)?
- [ ] Đã test bằng cách sinh chuỗi từ danh sách chữ số thật rồi xáo trộn?